# Session 1: a refrigerator moves heat uphill, and it needs work to do it
**A. Real-world problem.** Put a warm drink on the table and it cools down by itself.
Heat always flows from warm to cold. A refrigerator does the opposite: it keeps its inside
colder than the kitchen. Why does it have to be plugged in?

**Plan.** (1) learn the four parts, (2) write the energy balance, (3) define the COP
(coefficient of performance), (4) see that without work nothing moves.

Educational model: simplified numbers, not a design tool for real appliances.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

PARTS = [
    ("evaporator", "liquid boils at low pressure and takes heat from the inside"),
    ("compressor", "the electric motor squeezes the vapour; this is where the work W goes in"),
    ("condenser", "the hot compressed vapour gives heat to the kitchen and turns back to liquid"),
    ("expansion valve", "a narrow opening drops the pressure so the liquid becomes cold again"),
]

## Step 1: the energy balance
Over one full turn of the loop the fluid ends where it started, so it stores no energy.
Energy in must equal energy out:

    Q_hot = Q_cold + W

* `Q_cold` is the heat taken from the cold inside (kJ),
* `W` is the work put in by the compressor (kJ),
* `Q_hot` is the heat given to the warm kitchen (kJ).

In [ ]:
def heat_to_room(q_cold, work):
    """Energy balance of one cycle: Q_hot = Q_cold + W (all in the same unit)."""
    if q_cold < 0 or work < 0:
        raise ValueError("heat and work are amounts here, so they cannot be negative")
    return q_cold + work


def cop(q_cold, work):
    """Coefficient of performance = heat removed per unit of work. Needs work > 0."""
    if work <= 0:
        raise ValueError("with no work there is no refrigeration: COP is not defined")
    return q_cold / work


def cooling_from_work(work, cop_value):
    """Heat removed from the inside when `work` is used by a fridge with this COP."""
    return cop_value * work

## Step 2: how much must a fridge remove?
In steady state the inside stays at the same temperature, so the fridge must remove exactly
the heat that leaks in through the walls. Leak power = UA x (T_room - T_inside), where UA
(W per kelvin) says how leaky the walls are.

In [ ]:
def wall_leak_watts(ua, t_room, t_inside):
    """Heat leaking in through the walls, in watts (J per second)."""
    return ua * (t_room - t_inside)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # worked example numbers: 300 J removed using 100 J of work
    qc, w = 300.0, 100.0
    qh = heat_to_room(qc, w)
    print(f"Q_cold = {qc:.0f} J, W = {w:.0f} J -> Q_hot = {qh:.0f} J, COP = {cop(qc, w):.2f}")
    leak = wall_leak_watts(2.0, 25.0, 4.0)
    print(f"wall leak with UA = 2.0 W/K, 25 C kitchen, 4 C inside: {leak:.0f} W")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = ["Q_cold\n(from inside)", "W\n(compressor)", "Q_hot\n(to kitchen)"]
    ax.bar(names, [qc, w, qh], color=["tab:blue", "tab:orange", "tab:red"])
    for i, v in enumerate([qc, w, qh]):
        ax.text(i, v + 6, f"{v:.0f} J", ha="center")
    ax.set_ylabel("energy per cycle (J)")
    ax.set_title("Energy flows of one cycle: Q_hot = Q_cold + W")
    ax.set_ylim(0, 480)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "energy_flow.png", dpi=120)

    work = np.linspace(0, 5, 51)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for c, col in ((2.0, "tab:green"), (3.0, "tab:blue"), (4.0, "tab:purple")):
        ax2.plot(work, cooling_from_work(work, c), color=col, label=f"heat removed, COP = {c:.0f}")
    ax2.plot(work, work, "k--", label="work put in (W)")
    ax2.set_xlabel("work W (kJ)")
    ax2.set_ylabel("heat removed from inside, Q_cold (kJ)")
    ax2.set_title("No work, no cooling; more work, more heat moved")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "cooling_vs_work.png", dpi=120)

## Where this stops
We have not asked how large the COP can be. Session 2 answers: there is a limit set by the
two temperatures, and it is why a hot kitchen costs more to cool against.

## Coding tasks
1. **Easy.** Change the numbers in `lesson.py` to 450 J removed with 150 J of work. Predict Q_hot and the COP before you run it.
2. **Stretch.** Write `compressor_power(leak_w, cop_value)` that returns the watts needed to remove a steady heat leak, and test it with a leak of 42 W and COP 3.
3. **Explain.** Explain, using Q_hot = Q_cold + W, why a fridge in a closed, insulated room makes the room warmer when it runs.

## Assessment
- **A. Conceptual.** Why can a refrigerator move heat from a cold interior to a warmer room only when energy is supplied? Why does the room get warmer, not colder, if you leave a fridge door open in a closed kitchen?
- **B. Numerical.** A fridge takes 450 J from the inside using 150 J of work. Find Q_hot and the COP, with units and a sanity check.
- **C. Graph and simulation interpretation.** In the graph of heat removed against work, what does the slope of each line mean? What is the heat removed at W = 2 kJ for COP = 3, and at W = 0?
- **D. Design and optimisation challenge.** Design: the walls leak 42 W (UA = 2.0 W/K, 25 C kitchen, 4 C inside). You can choose a fridge with COP 2.5 or COP 3.5, and your supply allows at most 15 W for the compressor. Which choices work, what is the compressor power for each, and how much heat does each give to the kitchen?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    qh = L.heat_to_room(300.0, 100.0)
    checks.append((f"worked example: 300 J removed with 100 J of work gives Q_hot = 400 J (got {qh:.0f})", abs(qh - 400.0) < 1e-9))
    checks.append(("worked example: COP of that cycle is 3", abs(L.cop(300.0, 100.0) - 3.0) < 1e-9))
    checks.append(("Q_hot is always larger than Q_cold when work is used", all(L.heat_to_room(q, w) > q for q in (1, 5, 40) for w in (0.1, 1, 3))))
    checks.append(("heat removed = COP x work", abs(L.cooling_from_work(2.0, 3.0) - 6.0) < 1e-12))
    checks.append(("round trip: cop(cooling_from_work(w, c), w) = c", abs(L.cop(L.cooling_from_work(1.5, 2.5), 1.5) - 2.5) < 1e-12))
    checks.append(("hand calculation: leak of 2.0 W/K over 21 K is 42 W", abs(L.wall_leak_watts(2.0, 25.0, 4.0) - 42.0) < 1e-9))
    checks.append(("no leak when the room is as cold as the inside", L.wall_leak_watts(2.0, 4.0, 4.0) == 0.0))
    try:
        L.cop(10.0, 0.0)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge case: zero work gives an error, not a number", raised))
    try:
        L.heat_to_room(-1.0, 1.0)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge case: negative amounts are rejected", raised))
    checks.append(("four parts are listed", len(L.PARTS) == 4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
